In [ ]:
#!/usr/bin/env python3
"""
LandCover.ai Dataset Preparation for CNN Training
This script tiles large satellite images and their masks into training patches
"""

import glob
import os
import cv2
import numpy as np
from pathlib import Path

# Configuration - Updated paths
IMGS_DIR = "./LandCover/Images"
MASKS_DIR = "./LandCover/Masks"
OUTPUT_DIR = "./LandCover/output"
TARGET_SIZE = 512

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Get image files
img_paths = sorted(glob.glob(os.path.join(IMGS_DIR, "*.tif")))

print(f"Found {len(img_paths)} images")

# Exit if no images found
if len(img_paths) == 0:
    print(f"\n❌ ERROR: No .tif images found in {IMGS_DIR}")
    print(f"   Current working directory: {os.getcwd()}")
    print(f"   Please verify:")
    print(f"   1. Image files are in: {os.path.abspath(IMGS_DIR)}")
    print(f"   2. Mask files are in: {os.path.abspath(MASKS_DIR)}")
    exit(1)

# Match each image with its corresponding mask
matched_pairs = []
missing_masks = []

for img_path in img_paths:
    img_filename = os.path.splitext(os.path.basename(img_path))[0]
    
    # Try common mask naming conventions
    possible_mask_names = [
        f"{img_filename}.tif",           # Same name
        f"{img_filename}_m.tif",         # With _m suffix
        f"{img_filename}_mask.tif",      # With _mask suffix
    ]
    
    mask_path = None
    for mask_name in possible_mask_names:
        potential_path = os.path.join(MASKS_DIR, mask_name)
        if os.path.exists(potential_path):
            mask_path = potential_path
            break
    
    if mask_path:
        matched_pairs.append((img_path, mask_path))
    else:
        missing_masks.append(img_filename)

print(f"Successfully matched {len(matched_pairs)} image-mask pairs")

if missing_masks:
    print(f"\n WARNING: {len(missing_masks)} images have no matching masks:")
    for name in missing_masks[:5]:  # Show first 5
        print(f"   - {name}")
    if len(missing_masks) > 5:
        print(f"   ... and {len(missing_masks) - 5} more")
    print()

if len(matched_pairs) == 0:
    print("\n❌ ERROR: No matching image-mask pairs found!")
    exit(1)

# Process each image-mask pair
total_tiles = 0
for i, (img_path, mask_path) in enumerate(matched_pairs):
    img_filename = os.path.splitext(os.path.basename(img_path))[0]
    mask_filename = os.path.splitext(os.path.basename(mask_path))[0]
    
    # Read images
    img = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)  # Read mask as grayscale
    
    # Verify matching pairs (checks base filename without suffix)
    mask_base = mask_filename.replace('_m', '').replace('_mask', '')
    assert img_filename == mask_base, f"Mismatch: {img_filename} != {mask_base}"
    assert img.shape[:2] == mask.shape, f"Shape mismatch: {img.shape[:2]} != {mask.shape}"
    
    print(f"\nProcessing {img_filename} ({i+1}/{len(matched_pairs)})")
    print(f"  Image shape: {img.shape}, Mask shape: {mask.shape}")
    
    k = 0
    tiles_saved = 0
    
    # Tile the image with stride equal to TARGET_SIZE (no overlap)
    for y in range(0, img.shape[0], TARGET_SIZE):
        for x in range(0, img.shape[1], TARGET_SIZE):
            # Extract tiles
            img_tile = img[y:y + TARGET_SIZE, x:x + TARGET_SIZE]
            mask_tile = mask[y:y + TARGET_SIZE, x:x + TARGET_SIZE]
            
            # Only save complete tiles (512x512)
            if img_tile.shape[0] == TARGET_SIZE and img_tile.shape[1] == TARGET_SIZE:
                # Save image tile as JPEG
                out_img_path = os.path.join(OUTPUT_DIR, f"{img_filename}_{k}.jpg")
                cv2.imwrite(out_img_path, img_tile)
                
                # Save mask tile as PNG (lossless)
                out_mask_path = os.path.join(OUTPUT_DIR, f"{mask_filename}_{k}_m.png")
                cv2.imwrite(out_mask_path, mask_tile)
                
                tiles_saved += 1
            
            k += 1
    
    total_tiles += tiles_saved
    print(f"  Saved {tiles_saved} tiles")

print(f"\n✓ Complete! Total tiles saved: {total_tiles}")

# ============================================================================
# NEXT STEPS: Creating a PyTorch/TensorFlow Dataset
# ============================================================================

print("\n" + "="*70)
print("DATASET STRUCTURE:")
print("="*70)
print("""
Your output directory now contains:
- Image tiles: {filename}_{index}.jpg
- Mask tiles: {filename}_{index}_m.png

LandCover.ai Classes:
1. Background (0) - areas without objects
2. Buildings (1) - residential, commercial, industrial
3. Woodlands (2) - forests, tree lines
4. Water (3) - rivers, lakes, ponds
5. Roads (4) - highways, streets, paths

Next: Create a custom dataset loader (see example below)
""")

# ============================================================================
# EXAMPLE: PyTorch Dataset Loader
# ============================================================================

print("="*70)
print("PyTorch Dataset Example:")
print("="*70)
print("""
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch

class LandCoverDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        
        # Get all image files (not masks)
        self.img_files = sorted([f for f in os.listdir(img_dir) 
                                if f.endswith('.jpg')])
    
    def __len__(self):
        return len(self.img_files)
    
    def __getitem__(self, idx):
        # Load image
        img_name = self.img_files[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        
        # Load corresponding mask
        mask_name = img_name.replace('.jpg', '_m.png')
        mask_path = os.path.join(self.img_dir, mask_name)
        mask = Image.open(mask_path)
        mask = np.array(mask)
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        # Convert mask to tensor
        mask = torch.from_numpy(mask).long()
        
        return image, mask

# Usage:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])

dataset = LandCoverDataset('./LandCover/output', transform=transform)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
""")

In [ ]:
#4layer#

#!/usr/bin/env python3
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor, EarlyStopping, Callback
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from torchvision import models
import albumentations as A
from albumentations.pytorch import ToTensorV2
import time
from functools import wraps
import pandas as pd

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

# ============================================================
# TIMING UTILITIES
# ============================================================
class Timer:
    """Context manager and decorator for timing code blocks"""
    def __init__(self, name=""):
        self.name = name
        self.start_time = None
        self.elapsed = None
    
    def __enter__(self):
        self.start_time = time.time()
        print(f"⏱️  [{self.name}] Started at {time.strftime('%H:%M:%S')}")
        return self
    
    def __exit__(self, *args):
        self.elapsed = time.time() - self.start_time
        print(f" [{self.name}] Completed in {self.elapsed:.2f}s\n")

def timeit(func):
    """Decorator to time functions"""
    @wraps(func)
    def wrapper(*args, **kwargs):
        func_name = func.__name__
        start = time.time()
        print(f"⏱️  [{func_name}] Started at {time.strftime('%H:%M:%S')}")
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"✅ [{func_name}] Completed in {elapsed:.2f}s\n")
        return result
    return wrapper

# Global timing tracker
timing_stats = {}

def log_timing(name, duration):
    """Log timing statistics"""
    if name not in timing_stats:
        timing_stats[name] = []
    timing_stats[name].append(duration)

# ============================================================
# LIGHT AUGMENTATION TRANSFORMS
# ============================================================
def get_light_train_transform():
    """Light augmentation for training - subtle transformations"""
    return A.Compose([
        # Geometric transforms (light)
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.3),
        A.ShiftScaleRotate(
            shift_limit=0.05,  # Small shift
            scale_limit=0.05,  # Small scale
            rotate_limit=10,   # Small rotation
            p=0.3
        ),
        
        # Color/intensity adjustments (very light)
        A.RandomBrightnessContrast(
            brightness_limit=0.1,  # Subtle brightness
            contrast_limit=0.1,    # Subtle contrast
            p=0.3
        ),
        A.HueSaturationValue(
            hue_shift_limit=10,
            sat_shift_limit=15,
            val_shift_limit=10,
            p=0.2
        ),
        
        # Normalize and convert to tensor
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

def get_val_transform():
    """Validation transform - only normalization"""
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

# ============================================================
# 1️⃣ Dataset
# ============================================================
class LandCoverDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        print(f"⏱️  [Dataset Init] Started at {time.strftime('%H:%M:%S')}")
        start = time.time()
        
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))
        self.transform = transform
        
        elapsed = time.time() - start
        print(f"✅ [Dataset Init] Found {len(self.images)} images in {elapsed:.2f}s\n")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        start = time.time()
        
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name.replace(".jpg", "_m.png"))

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = np.clip(mask, 0, 4)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = torch.tensor(augmented['mask'], dtype=torch.long)
        else:
            image = image.astype(np.float32) / 255.0
            image = torch.tensor(image).permute(2, 0, 1)
            mask = torch.tensor(mask, dtype=torch.long)

        elapsed = time.time() - start
        log_timing('dataset_getitem', elapsed)
        
        return image, mask

# ============================================================
# 3️⃣ Compute class weights
# ============================================================
@timeit
def compute_class_weights(dataset, n_classes=5):
    counts = np.zeros(n_classes, dtype=np.float64)
    for i, (_, mask) in enumerate(dataset):
        if i % 20 == 0:
            print(f"   Processing sample {i+1}/{len(dataset)}")
        mask_np = mask.numpy().flatten()
        for c in range(n_classes):
            counts[c] += (mask_np == c).sum()
    freqs = counts / counts.sum()
    class_weights = 1.0 / (freqs + 1e-6)
    class_weights = class_weights / class_weights.sum() * n_classes
    return torch.tensor(class_weights, dtype=torch.float32)

# ============================================================
# 4️⃣ ResNet34 U-Net with 4 LAYERS + DROPOUT (UP TO 256 CHANNELS)
# ============================================================
class ResNet34UNet4Layer(nn.Module):
    """
    ResNet34 U-Net with 4 encoder layers (up to layer3)
    Uses layer3 with 256 channels as bottleneck
    WITH DROPOUT for better regularization
    """
    def __init__(self, n_classes=5, pretrained=True, dropout_p=0.2):
        super().__init__()

        # Load pre-trained ResNet34
        try:
            resnet = models.resnet34(pretrained=pretrained)
        except TypeError:
            # fallback for newer torchvision versions
            resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)

        # Encoder - Use first 4 layers
        self.encoder1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  # 64 channels
        self.pool = resnet.maxpool
        self.encoder2 = resnet.layer1  # 64 channels
        self.encoder3 = resnet.layer2  # 128 channels
        self.encoder4 = resnet.layer3  # 256 channels (BOTTLENECK)

        # Decoder - 3 upsampling stages
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = self._make_decoder_block(256, 128, dropout_p)  # concat (128 + 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = self._make_decoder_block(128, 64, dropout_p)   # concat (64 + 64)

        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec1 = self._make_decoder_block(128, 64, dropout_p)   # concat (64 + 64)

        # Final upsample to restore original resolution
        self.final_upsample = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.final = nn.Conv2d(64, n_classes, kernel_size=1)

    def _make_decoder_block(self, in_ch, out_ch, dropout_p):
        """Decoder block with dropout after each conv block"""
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout_p),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout_p)
        )

    def forward(self, x):
        # Encoder with skip connections
        e1 = self.encoder1(x)          # [B, 64, H/2, W/2] (conv1 has stride 2)
        e1_pooled = self.pool(e1)      # [B, 64, H/4, W/4]
        e2 = self.encoder2(e1_pooled)  # [B, 64, H/4, W/4]
        e3 = self.encoder3(e2)         # [B, 128, H/8, W/8]
        e4 = self.encoder4(e3)         # [B, 256, H/16, W/16] - BOTTLENECK

        # Decoder with skip connections
        d3 = self.up3(e4)                        # [B, 128, H/8, W/8]
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)                        # [B, 64, H/4, W/4]
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)                        # [B, 64, H/2, W/2]
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        d1 = self.final_upsample(d1)             # [B, 64, H, W]
        return self.final(d1)                    # [B, n_classes, H, W]


# ============================================================
# 5️⃣ Lightning Module
# ============================================================
class LitUNet(pl.LightningModule):
    def __init__(self, n_classes=5, lr=1e-3, class_weights=None, dropout_p=0.2, weight_decay=1e-4):
        super().__init__()
        self.save_hyperparameters(ignore=['class_weights'])
        
        self.model = ResNet34UNet4Layer(n_classes, pretrained=True, dropout_p=dropout_p)
        
        if class_weights is not None:
            self.criterion = nn.CrossEntropyLoss(weight=class_weights)
        else:
            self.criterion = nn.CrossEntropyLoss()
        
        self.lr = lr
        self.n_classes = n_classes
        self.dropout_p = dropout_p
        self.weight_decay = weight_decay
        
        # Timing trackers
        self.train_step_times = []
        self.val_step_times = []

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        step_start = time.time()
        
        imgs, masks = batch
        outputs = self(imgs)
        loss = self.criterion(outputs, masks)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        
        step_time = time.time() - step_start
        self.train_step_times.append(step_time)
        
        if batch_idx % 10 == 0:
            print(f"   Train batch {batch_idx}: {step_time:.3f}s")
        
        return loss

    def validation_step(self, batch, batch_idx):
        step_start = time.time()
        
        imgs, masks = batch
        outputs = self(imgs)
        loss = self.criterion(outputs, masks)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        preds = torch.argmax(outputs, dim=1)
        acc, miou, dice, _, _ = self.compute_metrics(preds, masks)
        self.log("val_acc", acc, prog_bar=True)
        self.log("val_miou", miou, prog_bar=True)
        self.log("val_dice", dice, prog_bar=True)
        
        step_time = time.time() - step_start
        self.val_step_times.append(step_time)
        
        if batch_idx % 5 == 0:
            print(f"   Val batch {batch_idx}: {step_time:.3f}s")
        
        return loss

    def on_train_epoch_end(self):
        if self.train_step_times:
            avg_time = np.mean(self.train_step_times)
            print(f"\n Train epoch avg step time: {avg_time:.3f}s")
            self.train_step_times = []

    def on_validation_epoch_end(self):
        if self.val_step_times:
            avg_time = np.mean(self.val_step_times)
            print(f" Val epoch avg step time: {avg_time:.3f}s\n")
            self.val_step_times = []

    def configure_optimizers(self):
        # Encoder layers (4 layers)
        encoder_params = (
            list(self.model.encoder1.parameters()) + 
            list(self.model.encoder2.parameters()) + 
            list(self.model.encoder3.parameters()) +
            list(self.model.encoder4.parameters())
        )
        
        # Decoder layers (3 levels)
        decoder_params = (
            list(self.model.up3.parameters()) + 
            list(self.model.dec3.parameters()) + 
            list(self.model.up2.parameters()) + 
            list(self.model.dec2.parameters()) + 
            list(self.model.up1.parameters()) + 
            list(self.model.dec1.parameters()) + 
            list(self.model.final_upsample.parameters()) + 
            list(self.model.final.parameters())
        )
        
        optimizer = torch.optim.Adam([
            {'params': encoder_params, 'lr': self.lr * 0.1},
            {'params': decoder_params, 'lr': self.lr}
        ], weight_decay=self.weight_decay)
    
        
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-7
        )
        
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1
            }
        }

    def compute_metrics(self, preds, masks):
        preds = preds.flatten().cpu().numpy()
        masks = masks.flatten().cpu().numpy()
        
        intersection = np.zeros(self.n_classes)
        union = np.zeros(self.n_classes)
        dice = np.zeros(self.n_classes)
        
        accuracy = (preds == masks).mean()
        
        for c in range(self.n_classes):
            pred_c = preds == c
            mask_c = masks == c
            inter = np.logical_and(pred_c, mask_c).sum()
            union_c = np.logical_or(pred_c, mask_c).sum()
            intersection[c] = inter
            union[c] = union_c
            dice[c] = (2 * inter) / (pred_c.sum() + mask_c.sum() + 1e-6)
        
        IoU = intersection / np.maximum(union, 1)
        mean_IoU = np.nanmean(IoU)
        mean_dice = np.nanmean(dice)
        
        return accuracy, mean_IoU, mean_dice, IoU, dice


# ============================================================
# 6️⃣ Visualization
# ============================================================
class_info = {
    0: ("Background", (0, 0, 0)),
    1: ("Building",   (255, 0, 0)),
    2: ("Woodland",   (0, 255, 0)),
    3: ("Water",      (0, 0, 255)),
    4: ("Road",       (255, 255, 0)),
}

def id_to_color(mask):
    color_mask = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for class_id, (label, color) in class_info.items():
        color_mask[mask == class_id] = color
    return color_mask

@timeit
def show_predictions(model, dataset, num_samples=30, save_dir=None):
    model.eval()
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)

    # Create save directory if specified
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)

    for i in range(num_samples):
        img, mask = dataset[i]
        if hasattr(dataset, "dataset"):
            img_name = os.path.basename(dataset.dataset.images[dataset.indices[i]])
        else:
            img_name = os.path.basename(dataset.images[i])

        with torch.no_grad():
            pred = model(img.unsqueeze(0))
        pred = torch.argmax(pred, dim=1).squeeze().cpu().numpy()
        mask_np = mask.cpu().numpy()

        # Denormalize image for visualization
        img_np = img.cpu().numpy().transpose(1, 2, 0)
        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)
        
        overlay_pred = id_to_color(pred)
        overlay_mask = id_to_color(mask_np)
        overlay_gt = cv2.addWeighted(img_np, 0.6, overlay_mask, 0.4, 0)
        overlay_pred_blend = cv2.addWeighted(img_np, 0.6, overlay_pred, 0.4, 0)

        unique, counts = np.unique(pred, return_counts=True)
        dominant_class = unique[np.argmax(counts)]
        dominant_label = class_info.get(dominant_class, ("Background", None))[0] \
                         if dominant_class != 0 else "Background"

        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title(f"{img_name}", fontsize=9)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(overlay_gt)
        axes[i, 1].set_title("Ground Truth", fontsize=9)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(overlay_pred_blend)
        axes[i, 2].set_title(f"Prediction ({dominant_label})", fontsize=9)
        axes[i, 2].axis('off')

        # Save each row as its own file
        if save_dir is not None:
            save_path = os.path.join(save_dir, f"{os.path.splitext(img_name)[0]}_prediction.png")
            row_fig, row_axes = plt.subplots(1, 3, figsize=(12, 4))
            row_axes[0].imshow(img_np)
            row_axes[0].set_title(f"{img_name}", fontsize=9)
            row_axes[0].axis('off')
            row_axes[1].imshow(overlay_gt)
            row_axes[1].set_title("Ground Truth", fontsize=9)
            row_axes[1].axis('off')
            row_axes[2].imshow(overlay_pred_blend)
            row_axes[2].set_title(f"Prediction ({dominant_label})", fontsize=9)
            row_axes[2].axis('off')
            plt.tight_layout()
            plt.savefig(save_path, bbox_inches="tight")
            plt.close(row_fig)
            print(f"Saved: {save_path}")

    legend_elements = [Patch(facecolor=np.array(color)/255.0, edgecolor='black', label=label)
                       for label, color in [v for v in class_info.values()]]
    fig.legend(handles=legend_elements, loc='upper right', title="Classes")
    plt.tight_layout()

    # Save the full figure too
    if save_dir is not None:
        combined_path = os.path.join(save_dir, "combined_predictions.png")
        plt.savefig(combined_path, bbox_inches="tight")
        print(f"\nSaved combined predictions grid to: {combined_path}\n")

    plt.show()


# ============================================================
# 7️⃣ Metrics Logger
# ============================================================
class MetricsLogger(Callback):
    def __init__(self):
        super().__init__()
        self.train_losses, self.val_losses = [], []
        self.val_accs, self.val_mious, self.val_dices = [], [], []
        self.epoch_times = []
        self.epoch_start = None

    def on_train_epoch_start(self, trainer, pl_module):
        self.epoch_start = time.time()
        print(f"\n{'='*70}")
        print(f"EPOCH {trainer.current_epoch + 1} - Started at {time.strftime('%H:%M:%S')}")
        print(f"{'='*70}")

    def on_train_epoch_end(self, trainer, pl_module):
        train_loss = trainer.callback_metrics.get('train_loss')
        if train_loss is not None:
            self.train_losses.append(train_loss.item())

    def on_validation_epoch_end(self, trainer, pl_module):
        val_loss = trainer.callback_metrics.get('val_loss')
        val_acc = trainer.callback_metrics.get('val_acc')
        val_miou = trainer.callback_metrics.get('val_miou')
        val_dice = trainer.callback_metrics.get('val_dice')
        if val_loss is not None: self.val_losses.append(val_loss.item())
        if val_acc is not None: self.val_accs.append(val_acc.item())
        if val_miou is not None: self.val_mious.append(val_miou.item())
        if val_dice is not None: self.val_dices.append(val_dice.item())
        
        if self.epoch_start:
            epoch_time = time.time() - self.epoch_start
            self.epoch_times.append(epoch_time)
            print(f"\n  Epoch {trainer.current_epoch + 1} total time: {epoch_time:.2f}s")
            print(f"{'='*70}\n")

@timeit
def plot_epoch_metrics(metrics_logger, lr, batch_size, dropout_p, csv_filename='training_metrics_fulltrainingset_4layers.csv'):
    min_len = min(
        len(metrics_logger.train_losses), len(metrics_logger.val_losses),
        len(metrics_logger.val_accs), len(metrics_logger.val_mious),
        len(metrics_logger.val_dices)
    )
    epochs = range(1, min_len + 1)

    # Save metrics to CSV
    data = {
        "Epoch": list(epochs),
        "Train_Loss": metrics_logger.train_losses[:min_len],
        "Val_Loss": metrics_logger.val_losses[:min_len],
        "Val_Accuracy": metrics_logger.val_accs[:min_len],
        "Val_mIoU": metrics_logger.val_mious[:min_len],
        "Val_Dice": metrics_logger.val_dices[:min_len]
    }
    df = pd.DataFrame(data)
    df.to_csv(csv_filename, index=False)
    print(f" Training metrics saved to {csv_filename}")
    
    # Plotting
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes_flat = axes.flatten()

    # Loss
    axes_flat[0].plot(epochs, metrics_logger.train_losses[:min_len], 'b-o', label='Train Loss', markersize=3)
    axes_flat[0].plot(epochs, metrics_logger.val_losses[:min_len], 'r-o', label='Val Loss', markersize=3)
    axes_flat[0].set_title('Loss')
    axes_flat[0].set_xlabel('Epoch')
    axes_flat[0].set_ylabel('Loss')
    axes_flat[0].legend()
    axes_flat[0].grid(True)

    # Accuracy
    axes_flat[1].plot(epochs, metrics_logger.val_accs[:min_len], 'g-o', markersize=3)
    axes_flat[1].set_title('Accuracy')
    axes_flat[1].set_xlabel('Epoch')
    axes_flat[1].set_ylabel('Accuracy')
    axes_flat[1].grid(True)

    # Mean IoU
    axes_flat[2].plot(epochs, metrics_logger.val_mious[:min_len], 'm-o', label='Val mIoU', markersize=3)
    axes_flat[2].set_title('Mean IoU')
    axes_flat[2].set_xlabel('Epoch')
    axes_flat[2].set_ylabel('mIoU')
    axes_flat[2].legend()
    axes_flat[2].grid(True)
    best_miou = max(metrics_logger.val_mious[:min_len])
    axes_flat[2].axhline(y=best_miou, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_miou:.4f}')
    axes_flat[2].legend()

    # Dice Score
    axes_flat[3].plot(epochs, metrics_logger.val_dices[:min_len], 'c-o', markersize=3)
    axes_flat[3].set_title('Dice Score')
    axes_flat[3].set_xlabel('Epoch')
    axes_flat[3].set_ylabel('Dice')
    axes_flat[3].grid(True)

    fig.suptitle(f'ResNet34 4-Layer Full Training Set', 
                 fontsize=20, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'training_metrics_dropout_{dropout_p}_light_aug_full_training_set_4layer.png', dpi=150, bbox_inches='tight')
    plt.show()


# ============================================================
# 8️⃣ Helper - OPTIMIZED VERSION
# ============================================================
import pickle
import hashlib
from pathlib import Path

def get_dataset_hash(img_dir, mask_dir):
    """Create a hash from directory paths to use as cache key"""
    combined = f"{img_dir}_{mask_dir}"
    return hashlib.md5(combined.encode()).hexdigest()[:8]

def scan_dataset_classes(dataset, n_classes=5, cache_dir="cache"):
    """Scan dataset once and cache results"""
    # Create cache directory
    os.makedirs(cache_dir, exist_ok=True)
    
    # Generate cache filename
    img_dir = dataset.img_dir if hasattr(dataset, 'img_dir') else dataset.dataset.img_dir
    mask_dir = dataset.mask_dir if hasattr(dataset, 'mask_dir') else dataset.dataset.mask_dir
    cache_hash = get_dataset_hash(img_dir, mask_dir)
    cache_file = os.path.join(cache_dir, f"class_indices_{cache_hash}.pkl")
    
    # Try to load from cache
    if os.path.exists(cache_file):
        print(f"    Loading cached class information from {cache_file}")
        with open(cache_file, 'rb') as f:
            return pickle.load(f)
    
    # Scan dataset
    print(f"    Scanning dataset (this will be cached)...")
    class_to_indices = {c: [] for c in range(n_classes)}
    
    for idx in range(len(dataset)):
        if idx % 100 == 0:
            print(f"      Progress: {idx}/{len(dataset)}")
        
        # Load only the mask (much faster than loading through __getitem__)
        if hasattr(dataset, 'images'):
            img_name = dataset.images[idx]
            mask_path = os.path.join(dataset.mask_dir, img_name.replace(".jpg", "_m.png"))
        else:
            img_name = dataset.dataset.images[dataset.indices[idx]]
            mask_path = os.path.join(dataset.dataset.mask_dir, img_name.replace(".jpg", "_m.png"))
        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = np.clip(mask, 0, n_classes-1)
        
        # Check which classes are present
        unique_classes = np.unique(mask)
        for c in unique_classes:
            class_to_indices[c].append(idx)
    
    # Save to cache
    with open(cache_file, 'wb') as f:
        pickle.dump(class_to_indices, f)
    print(f"    Cached class information to {cache_file}")
    
    return class_to_indices

@timeit
def select_balanced_subset(dataset, n_samples=200, n_classes=5, use_cache=True):
    """Fast balanced subset selection using cached class information"""
    
    if use_cache:
        class_to_indices = scan_dataset_classes(dataset, n_classes)
    else:
        # Original slow method
        class_to_indices = {c: [] for c in range(n_classes)}
        for idx, (_, mask) in enumerate(dataset):
            if idx % 50 == 0:
                print(f"   Scanning sample {idx}/{len(dataset)}")
            mask_np = mask.numpy()
            for c in range(n_classes):
                if (mask_np == c).any():
                    class_to_indices[c].append(idx)
    
    # Select at least one sample from each class
    selected_indices = set()
    for c in range(n_classes):
        if class_to_indices[c]:
            selected_indices.add(class_to_indices[c][0])
            print(f"   Class {c}: {len(class_to_indices[c])} samples available")
    
    # Fill remaining with random samples
    all_indices = set(range(len(dataset)))
    remaining = list(all_indices - selected_indices)
    np.random.shuffle(remaining)
    while len(selected_indices) < n_samples and remaining:
        selected_indices.add(remaining.pop())
    
    print(f"   Selected {len(selected_indices)} samples")
    selected_indices = sorted(list(selected_indices))
    subset = torch.utils.data.Subset(dataset, selected_indices)
    return subset

def print_timing_summary():
    """Print summary of all timing statistics"""
    print("\n" + "="*70)
    print("TIMING SUMMARY")
    print("="*70)
    
    if 'dataset_getitem' in timing_stats:
        times = timing_stats['dataset_getitem']
        print(f"\nDataset __getitem__ calls: {len(times)}")
        print(f"  Avg: {np.mean(times):.4f}s")
        print(f"  Min: {np.min(times):.4f}s")
        print(f"  Max: {np.max(times):.4f}s")
        print(f"  Total: {np.sum(times):.2f}s")
    
    print("\n" + "="*70)

# ============================================================
# 9️⃣ Main - WITH EARLY STOPPING & LIGHT AUGMENTATION
# ============================================================
if __name__ == "__main__":
    total_start = time.time()
    print("\n" + "="*70)
    print(f"TRAINING STARTED AT {time.strftime('%H:%M:%S')}")
    print("="*70 + "\n")
    
    #  CONFIGURATION
    DROPOUT = 0.2
    WEIGHT_DECAY = 0.0001
    LEARNING_RATE = 1e-3
    BATCH_SIZE = 16
    EARLY_STOP_PATIENCE = 10  # Stop if no improvement for 10 epochs
    
    
    # Create datasets WITH light augmentation
    with Timer("Creating Train Dataset"):
        train_transform = get_light_train_transform()
        train_dataset = LandCoverDataset("data/train/images", "data/train/masks", transform=train_transform)
    
    with Timer("Creating Val Dataset"):
        val_transform = get_val_transform()
        val_dataset = LandCoverDataset("data/val/images", "data/val/masks", transform=val_transform)
    
    # Create temporary datasets without augmentation for class weight computation
    with Timer("Creating temporary dataset for class weights"):
        temp_dataset = LandCoverDataset("data/train/images", "data/train/masks", transform=None)
    
    # Select balanced subsets
    train_subset = select_balanced_subset(temp_dataset, n_samples=7470, n_classes=5)
    val_subset = select_balanced_subset(val_dataset, n_samples=1602, n_classes=5)
    
    # Compute class weights from raw data (without augmentation)
    class_weights = compute_class_weights(train_subset, n_classes=5)
    print(f"Class weights: {class_weights}\n")

    # Now create final subsets with augmentation using the same indices
    with Timer("Creating Final Dataset Subsets with Augmentation"):
        train_subset_final = torch.utils.data.Subset(train_dataset, train_subset.indices)
        val_subset_final = torch.utils.data.Subset(val_dataset, val_subset.indices)
    
    with Timer("Creating DataLoaders"):
        train_loader = DataLoader(train_subset_final, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
        val_loader   = DataLoader(val_subset_final, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    with Timer("Creating Model"):
        model = LitUNet(
            n_classes=5, 
            lr=LEARNING_RATE, 
            class_weights=class_weights.cuda() if torch.cuda.is_available() else class_weights,
            dropout_p=DROPOUT,
            weight_decay=WEIGHT_DECAY
        )

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,} ({total_params/1e6:.2f}M)")
    print(f"Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.2f}M)")
    print(f"\n HYPERPARAMETERS:")
    print(f"   Dropout: {DROPOUT}")
    print(f"   Weight Decay: {WEIGHT_DECAY}")
    print(f"   Learning Rate: {LEARNING_RATE}")
    print(f"   Batch Size: {BATCH_SIZE}\n")

    # Callbacks
    lr_monitor = LearningRateMonitor(logging_interval='epoch')
    metrics_logger = MetricsLogger()
    
    # Checkpoint callback - save best model based on val_miou
    checkpoint_callback = ModelCheckpoint(
        dirpath='checkpoints/',
        filename='resnet34-4layer-dropout0.2-wd0.0001-lr1e-3-{epoch:02d}-{val_miou:.4f}',
        monitor='val_miou',
        mode='max',
        save_top_k=3,
        verbose=True
    )
    
    #  EARLY STOPPING CALLBACK
    early_stop_callback = EarlyStopping(
        monitor='val_miou',      # Monitor validation mIoU
        min_delta=0.0001,        # Minimum change to qualify as improvement
        patience=EARLY_STOP_PATIENCE,  # Number of epochs with no improvement
        verbose=True,
        mode='max'               # We want to maximize mIoU
    )
    
    # Accelerator
    accelerator = "gpu" if torch.cuda.is_available() else "cpu"
    print("Using accelerator:", accelerator)
    
    with Timer("Creating Trainer"):
        trainer = pl.Trainer(
            max_epochs=50,
            accelerator=accelerator,
            precision="16-mixed",
            callbacks=[lr_monitor, metrics_logger, checkpoint_callback, early_stop_callback],
            log_every_n_steps=100,
            enable_checkpointing=True
        )

    with Timer("Training"):
        trainer.fit(model, train_loader, val_loader)

    # Results
    print("\n" + "="*70)
    print("TRAINING RESULTS")
    print("="*70)
    
    best_miou = max(metrics_logger.val_mious) if metrics_logger.val_mious else 0
    best_acc = max(metrics_logger.val_accs) if metrics_logger.val_accs else 0
    best_dice = max(metrics_logger.val_dices) if metrics_logger.val_dices else 0
    
    print(f"\nResNet34 4-Layer Configuration:")
    print(f"   Dropout:       {DROPOUT}")
    print(f"   Weight Decay:  {WEIGHT_DECAY}")
    print(f"   Learning Rate: {LEARNING_RATE}")
    print(f"   Batch Size:    {BATCH_SIZE}")
    print(f"   Best mIoU:     {best_miou:.4f}")
    print(f"   Best Accuracy: {best_acc:.4f}")
    print(f"   Best Dice:     {best_dice:.4f}")
    
    # Check if early stopping was triggered
    if early_stop_callback.stopped_epoch > 0:
        print(f"\n  Early stopping triggered at epoch {early_stop_callback.stopped_epoch + 1}")
        print(f"   Training stopped after {len(metrics_logger.val_mious)} epochs")
    else:
        print(f"\n Training completed all epochs without early stopping")

    print("\n" + "="*70)
    print("Metrics per epoch (last 10):")
    print("="*70)
    min_len = min(len(metrics_logger.train_losses), len(metrics_logger.val_losses),
                  len(metrics_logger.val_accs), len(metrics_logger.val_mious),
                  len(metrics_logger.val_dices))
    start_idx = max(0, min_len - 10)
    for i in range(start_idx, min_len):
        tl = metrics_logger.train_losses[i]
        vl = metrics_logger.val_losses[i]
        acc = metrics_logger.val_accs[i]
        miou = metrics_logger.val_mious[i]
        dice = metrics_logger.val_dices[i]
        epoch_time = metrics_logger.epoch_times[i] if i < len(metrics_logger.epoch_times) else 0
        print(f"Epoch {i+1:3d}: Train={tl:.4f}, Val={vl:.4f}, Acc={acc:.4f}, mIoU={miou:.4f}, Dice={dice:.4f}, Time={epoch_time:.1f}s")

    # Plot and save metrics
    plot_epoch_metrics(metrics_logger, lr=LEARNING_RATE, batch_size=BATCH_SIZE, dropout_p=DROPOUT)
    
    # Show some predictions
    print("\n" + "="*70)
    print("GENERATING SAMPLE PREDICTIONS")
    print("="*70 + "\n")
    output_dir = "predictions_4layer"
    show_predictions(model, val_subset_final, num_samples=30, save_dir=output_dir)

    
    # Print timing summary
    print_timing_summary()
    
    total_time = time.time() - total_start
    print("\n" + "="*70)
    print(f"TOTAL EXECUTION TIME: {total_time:.2f}s ({total_time/60:.2f} minutes)")
    print(f"FINISHED AT {time.strftime('%H:%M:%S')}")
    print("="*70)
    
    # Print best checkpoint info
    if checkpoint_callback.best_model_path:
        print(f"\n Best model saved at: {checkpoint_callback.best_model_path}")
        print(f"   Best mIoU: {checkpoint_callback.best_model_score:.4f}")
    
    # Summary of improvements
    print("\n" + "="*70)
    print(" MODEL ARCHITECTURE: 4-LAYER ResNet34 U-Net")
  

In [ ]:

#3layerfull#

#!/usr/bin/env python3
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor, EarlyStopping, Callback
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from torchvision import models
import albumentations as A
from albumentations.pytorch import ToTensorV2
import time
from functools import wraps
import pandas as pd

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

# ============================================================
# TIMING UTILITIES
# ============================================================
class Timer:
    """Context manager and decorator for timing code blocks"""
    def __init__(self, name=""):
        self.name = name
        self.start_time = None
        self.elapsed = None
    
    def __enter__(self):
        self.start_time = time.time()
        print(f"⏱️  [{self.name}] Started at {time.strftime('%H:%M:%S')}")
        return self
    
    def __exit__(self, *args):
        self.elapsed = time.time() - self.start_time
        print(f" [{self.name}] Completed in {self.elapsed:.2f}s\n")

def timeit(func):
    """Decorator to time functions"""
    @wraps(func)
    def wrapper(*args, **kwargs):
        func_name = func.__name__
        start = time.time()
        print(f"⏱️  [{func_name}] Started at {time.strftime('%H:%M:%S')}")
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f" [{func_name}] Completed in {elapsed:.2f}s\n")
        return result
    return wrapper

# Global timing tracker
timing_stats = {}

def log_timing(name, duration):
    """Log timing statistics"""
    if name not in timing_stats:
        timing_stats[name] = []
    timing_stats[name].append(duration)

# ============================================================
# LIGHT AUGMENTATION TRANSFORMS
# ============================================================
def get_light_train_transform():
    """Light augmentation for training - subtle transformations"""
    return A.Compose([
        # Geometric transforms (light)
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.3),
        A.ShiftScaleRotate(
            shift_limit=0.05,  # Small shift
            scale_limit=0.05,  # Small scale
            rotate_limit=10,   # Small rotation
            p=0.3
        ),
        
        # Color/intensity adjustments (very light)
        A.RandomBrightnessContrast(
            brightness_limit=0.1,  # Subtle brightness
            contrast_limit=0.1,    # Subtle contrast
            p=0.3
        ),
        A.HueSaturationValue(
            hue_shift_limit=10,
            sat_shift_limit=15,
            val_shift_limit=10,
            p=0.2
        ),
        
        # Normalize and convert to tensor
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

def get_val_transform():
    """Validation transform - only normalization"""
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

# ============================================================
# 1️⃣ Dataset
# ============================================================
class LandCoverDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        print(f"  [Dataset Init] Started at {time.strftime('%H:%M:%S')}")
        start = time.time()
        
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))
        self.transform = transform
        
        elapsed = time.time() - start
        print(f" [Dataset Init] Found {len(self.images)} images in {elapsed:.2f}s\n")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        start = time.time()
        
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name.replace(".jpg", "_m.png"))

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = np.clip(mask, 0, 4)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = torch.tensor(augmented['mask'], dtype=torch.long)
        else:
            image = image.astype(np.float32) / 255.0
            image = torch.tensor(image).permute(2, 0, 1)
            mask = torch.tensor(mask, dtype=torch.long)

        elapsed = time.time() - start
        log_timing('dataset_getitem', elapsed)
        
        return image, mask

# ============================================================
# 3️⃣ Compute class weights
# ============================================================
@timeit
def compute_class_weights(dataset, n_classes=5):
    counts = np.zeros(n_classes, dtype=np.float64)
    for i, (_, mask) in enumerate(dataset):
        if i % 20 == 0:
            print(f"   Processing sample {i+1}/{len(dataset)}")
        mask_np = mask.numpy().flatten()
        for c in range(n_classes):
            counts[c] += (mask_np == c).sum()
    freqs = counts / counts.sum()
    class_weights = 1.0 / (freqs + 1e-6)
    class_weights = class_weights / class_weights.sum() * n_classes
    return torch.tensor(class_weights, dtype=torch.float32)

# ============================================================
# 4️⃣ ResNet34 U-Net with 3 LAYERS + DROPOUT
# ============================================================
class ResNet34UNet3Layer(nn.Module):
    """
    ResNet34 U-Net with 3 encoder layers (up to layer2)
    Uses layer2 with 128 channels as bottleneck
    WITH DROPOUT for better regularization
    """
    def __init__(self, n_classes=5, pretrained=True, dropout_p=0.2):
        super().__init__()

        # Load pre-trained ResNet34
        try:
            resnet = models.resnet34(pretrained=pretrained)
        except TypeError:
            # fallback for newer torchvision versions
            resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)

        # Encoder - Only use first 3 layers
        self.encoder1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  # 64 channels
        self.pool = resnet.maxpool
        self.encoder2 = resnet.layer1  # 64 channels
        self.encoder3 = resnet.layer2  # 128 channels (BOTTLENECK)

        # Decoder - 2 upsampling stages
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = self._make_decoder_block(128, 64, dropout_p)  # concat (64 + 64)

        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec1 = self._make_decoder_block(128, 64, dropout_p)   # concat (64 + 64)

        # Final upsample to restore original resolution
        self.final_upsample = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.final = nn.Conv2d(64, n_classes, kernel_size=1)

    def _make_decoder_block(self, in_ch, out_ch, dropout_p):
        """Decoder block with dropout after each conv block"""
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout_p),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout_p)
        )

    def forward(self, x):
        # Encoder with skip connections
        e1 = self.encoder1(x)          # [B, 64, H/2, W/2] (conv1 has stride 2)
        e1_pooled = self.pool(e1)      # [B, 64, H/4, W/4]
        e2 = self.encoder2(e1_pooled)  # [B, 64, H/4, W/4]
        e3 = self.encoder3(e2)         # [B, 128, H/8, W/8] - BOTTLENECK

        # Decoder with skip connections
        d2 = self.up2(e3)                        # [B, 64, H/4, W/4]
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)                        # [B, 64, H/2, W/2]
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        d1 = self.final_upsample(d1)             # [B, 64, H, W]
        return self.final(d1)                    # [B, n_classes, H, W]


# ============================================================
# 5️⃣ Lightning Module
# ============================================================
class LitUNet(pl.LightningModule):
    def __init__(self, n_classes=5, lr=5e-4, class_weights=None, dropout_p=0.4, weight_decay=1e-4):
        super().__init__()
        self.save_hyperparameters(ignore=['class_weights'])
        
        self.model = ResNet34UNet3Layer(n_classes, pretrained=True, dropout_p=dropout_p)
        
        if class_weights is not None:
            self.criterion = nn.CrossEntropyLoss(weight=class_weights)
        else:
            self.criterion = nn.CrossEntropyLoss()
        
        self.lr = lr
        self.n_classes = n_classes
        self.dropout_p = dropout_p
        self.weight_decay = weight_decay
        
        # Timing trackers
        self.train_step_times = []
        self.val_step_times = []

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        step_start = time.time()
        
        imgs, masks = batch
        outputs = self(imgs)
        loss = self.criterion(outputs, masks)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        
        step_time = time.time() - step_start
        self.train_step_times.append(step_time)
        
        if batch_idx % 10 == 0:
            print(f"   Train batch {batch_idx}: {step_time:.3f}s")
        
        return loss

    def validation_step(self, batch, batch_idx):
        step_start = time.time()
        
        imgs, masks = batch
        outputs = self(imgs)
        loss = self.criterion(outputs, masks)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        preds = torch.argmax(outputs, dim=1)
        acc, miou, dice, _, _ = self.compute_metrics(preds, masks)
        self.log("val_acc", acc, prog_bar=True)
        self.log("val_miou", miou, prog_bar=True)
        self.log("val_dice", dice, prog_bar=True)
        
        step_time = time.time() - step_start
        self.val_step_times.append(step_time)
        
        if batch_idx % 5 == 0:
            print(f"   Val batch {batch_idx}: {step_time:.3f}s")
        
        return loss

    def on_train_epoch_end(self):
        if self.train_step_times:
            avg_time = np.mean(self.train_step_times)
            print(f"\n Train epoch avg step time: {avg_time:.3f}s")
            self.train_step_times = []

    def on_validation_epoch_end(self):
        if self.val_step_times:
            avg_time = np.mean(self.val_step_times)
            print(f" Val epoch avg step time: {avg_time:.3f}s\n")
            self.val_step_times = []

    def configure_optimizers(self):
        # Encoder layers (3 layers)
        encoder_params = (
            list(self.model.encoder1.parameters()) + 
            list(self.model.encoder2.parameters()) + 
            list(self.model.encoder3.parameters())
        )
        
        # Decoder layers (2 levels)
        decoder_params = (
            list(self.model.up2.parameters()) + 
            list(self.model.dec2.parameters()) + 
            list(self.model.up1.parameters()) + 
            list(self.model.dec1.parameters()) + 
            list(self.model.final_upsample.parameters()) + 
            list(self.model.final.parameters())
        )
        
        optimizer = torch.optim.Adam([
            {'params': encoder_params, 'lr': self.lr * 0.1},
            {'params': decoder_params, 'lr': self.lr}
        ], weight_decay=self.weight_decay)
    
        
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-7
        )
        
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1
            }
        }

    def compute_metrics(self, preds, masks):
        preds = preds.flatten().cpu().numpy()
        masks = masks.flatten().cpu().numpy()
        
        intersection = np.zeros(self.n_classes)
        union = np.zeros(self.n_classes)
        dice = np.zeros(self.n_classes)
        
        accuracy = (preds == masks).mean()
        
        for c in range(self.n_classes):
            pred_c = preds == c
            mask_c = masks == c
            inter = np.logical_and(pred_c, mask_c).sum()
            union_c = np.logical_or(pred_c, mask_c).sum()
            intersection[c] = inter
            union[c] = union_c
            dice[c] = (2 * inter) / (pred_c.sum() + mask_c.sum() + 1e-6)
        
        IoU = intersection / np.maximum(union, 1)
        mean_IoU = np.nanmean(IoU)
        mean_dice = np.nanmean(dice)
        
        return accuracy, mean_IoU, mean_dice, IoU, dice


# ============================================================
# 6️⃣ Visualisation
# ============================================================
class_info = {
    0: ("Background", (0, 0, 0)),
    1: ("Building",   (255, 0, 0)),
    2: ("Woodland",   (0, 255, 0)),
    3: ("Water",      (0, 0, 255)),
    4: ("Road",       (255, 255, 0)),
}

def id_to_color(mask):
    color_mask = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for class_id, (label, color) in class_info.items():
        color_mask[mask == class_id] = color
    return color_mask

@timeit
def show_predictions(model, dataset, num_samples=30, save_dir=None):
    model.eval()
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)

    # Create save directory if specified
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)

    for i in range(num_samples):
        img, mask = dataset[i]
        if hasattr(dataset, "dataset"):
            img_name = os.path.basename(dataset.dataset.images[dataset.indices[i]])
        else:
            img_name = os.path.basename(dataset.images[i])

        with torch.no_grad():
            pred = model(img.unsqueeze(0))
        pred = torch.argmax(pred, dim=1).squeeze().cpu().numpy()
        mask_np = mask.cpu().numpy()

        # Denormalize image for visualization
        img_np = img.cpu().numpy().transpose(1, 2, 0)
        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)
        
        overlay_pred = id_to_color(pred)
        overlay_mask = id_to_color(mask_np)
        overlay_gt = cv2.addWeighted(img_np, 0.6, overlay_mask, 0.4, 0)
        overlay_pred_blend = cv2.addWeighted(img_np, 0.6, overlay_pred, 0.4, 0)

        unique, counts = np.unique(pred, return_counts=True)
        dominant_class = unique[np.argmax(counts)]
        dominant_label = class_info.get(dominant_class, ("Background", None))[0] \
                         if dominant_class != 0 else "Background"

        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title(f"{img_name}", fontsize=9)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(overlay_gt)
        axes[i, 1].set_title("Ground Truth", fontsize=9)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(overlay_pred_blend)
        axes[i, 2].set_title(f"Prediction ({dominant_label})", fontsize=9)
        axes[i, 2].axis('off')

        # Save each row as its own file
        if save_dir is not None:
            save_path = os.path.join(save_dir, f"{os.path.splitext(img_name)[0]}_prediction.png")
            row_fig, row_axes = plt.subplots(1, 3, figsize=(12, 4))
            row_axes[0].imshow(img_np)
            row_axes[0].set_title(f"{img_name}", fontsize=9)
            row_axes[0].axis('off')
            row_axes[1].imshow(overlay_gt)
            row_axes[1].set_title("Ground Truth", fontsize=9)
            row_axes[1].axis('off')
            row_axes[2].imshow(overlay_pred_blend)
            row_axes[2].set_title(f"Prediction ({dominant_label})", fontsize=9)
            row_axes[2].axis('off')
            plt.tight_layout()
            plt.savefig(save_path, bbox_inches="tight")
            plt.close(row_fig)
            print(f"Saved: {save_path}")

    legend_elements = [Patch(facecolor=np.array(color)/255.0, edgecolor='black', label=label)
                       for label, color in [v for v in class_info.values()]]
    fig.legend(handles=legend_elements, loc='upper right', title="Classes")
    plt.tight_layout()

    # Save the full figure too
    if save_dir is not None:
        combined_path = os.path.join(save_dir, "combined_predictions.png")
        plt.savefig(combined_path, bbox_inches="tight")
        print(f"\nSaved combined predictions grid to: {combined_path}\n")

    plt.show()


# ============================================================
# 7️⃣ Metrics Logger
# ============================================================
class MetricsLogger(Callback):
    def __init__(self):
        super().__init__()
        self.train_losses, self.val_losses = [], []
        self.val_accs, self.val_mious, self.val_dices = [], [], []
        self.epoch_times = []
        self.epoch_start = None

    def on_train_epoch_start(self, trainer, pl_module):
        self.epoch_start = time.time()
        print(f"\n{'='*70}")
        print(f"EPOCH {trainer.current_epoch + 1} - Started at {time.strftime('%H:%M:%S')}")
        print(f"{'='*70}")

    def on_train_epoch_end(self, trainer, pl_module):
        train_loss = trainer.callback_metrics.get('train_loss')
        if train_loss is not None:
            self.train_losses.append(train_loss.item())

    def on_validation_epoch_end(self, trainer, pl_module):
        val_loss = trainer.callback_metrics.get('val_loss')
        val_acc = trainer.callback_metrics.get('val_acc')
        val_miou = trainer.callback_metrics.get('val_miou')
        val_dice = trainer.callback_metrics.get('val_dice')
        if val_loss is not None: self.val_losses.append(val_loss.item())
        if val_acc is not None: self.val_accs.append(val_acc.item())
        if val_miou is not None: self.val_mious.append(val_miou.item())
        if val_dice is not None: self.val_dices.append(val_dice.item())
        
        if self.epoch_start:
            epoch_time = time.time() - self.epoch_start
            self.epoch_times.append(epoch_time)
            print(f"\n Epoch {trainer.current_epoch + 1} total time: {epoch_time:.2f}s")
            print(f"{'='*70}\n")

@timeit
def plot_epoch_metrics(metrics_logger, lr, batch_size, dropout_p, csv_filename='training_metrics_fulltrainingset_3layers.csv'):
    min_len = min(
        len(metrics_logger.train_losses), len(metrics_logger.val_losses),
        len(metrics_logger.val_accs), len(metrics_logger.val_mious),
        len(metrics_logger.val_dices)
    )
    epochs = range(1, min_len + 1)

    # Save metrics to CSV
    data = {
        "Epoch": list(epochs),
        "Train_Loss": metrics_logger.train_losses[:min_len],
        "Val_Loss": metrics_logger.val_losses[:min_len],
        "Val_Accuracy": metrics_logger.val_accs[:min_len],
        "Val_mIoU": metrics_logger.val_mious[:min_len],
        "Val_Dice": metrics_logger.val_dices[:min_len]
    }
    df = pd.DataFrame(data)
    df.to_csv(csv_filename, index=False)
    print(f" Training metrics saved to {csv_filename}")
    
    # Plotting
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes_flat = axes.flatten()

    # Loss
    axes_flat[0].plot(epochs, metrics_logger.train_losses[:min_len], 'b-o', label='Train Loss', markersize=3)
    axes_flat[0].plot(epochs, metrics_logger.val_losses[:min_len], 'r-o', label='Val Loss', markersize=3)
    axes_flat[0].set_title('Loss')
    axes_flat[0].set_xlabel('Epoch')
    axes_flat[0].set_ylabel('Loss')
    axes_flat[0].legend()
    axes_flat[0].grid(True)

    # Accuracy
    axes_flat[1].plot(epochs, metrics_logger.val_accs[:min_len], 'g-o', markersize=3)
    axes_flat[1].set_title('Accuracy')
    axes_flat[1].set_xlabel('Epoch')
    axes_flat[1].set_ylabel('Accuracy')
    axes_flat[1].grid(True)

    # Mean IoU
    axes_flat[2].plot(epochs, metrics_logger.val_mious[:min_len], 'm-o', label='Val mIoU', markersize=3)
    axes_flat[2].set_title('Mean IoU')
    axes_flat[2].set_xlabel('Epoch')
    axes_flat[2].set_ylabel('mIoU')
    axes_flat[2].legend()
    axes_flat[2].grid(True)
    best_miou = max(metrics_logger.val_mious[:min_len])
    axes_flat[2].axhline(y=best_miou, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_miou:.4f}')
    axes_flat[2].legend()

    # Dice Score
    axes_flat[3].plot(epochs, metrics_logger.val_dices[:min_len], 'c-o', markersize=3)
    axes_flat[3].set_title('Dice Score')
    axes_flat[3].set_xlabel('Epoch')
    axes_flat[3].set_ylabel('Dice')
    axes_flat[3].grid(True)

    fig.suptitle(f'ResNet34 3-Layer Full Training Set', 
                 fontsize=20, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'training_metrics_dropout_{dropout_p}_light_aug_full_training_set_3layer.png', dpi=150, bbox_inches='tight')
    plt.show()


# ============================================================
# 8️⃣ Helper 
# ============================================================
import pickle
import hashlib
from pathlib import Path

def get_dataset_hash(img_dir, mask_dir):
    """Create a hash from directory paths to use as cache key"""
    combined = f"{img_dir}_{mask_dir}"
    return hashlib.md5(combined.encode()).hexdigest()[:8]

def scan_dataset_classes(dataset, n_classes=5, cache_dir="cache"):
    """Scan dataset once and cache results"""
    # Create cache directory
    os.makedirs(cache_dir, exist_ok=True)
    
    # Generate cache filename
    img_dir = dataset.img_dir if hasattr(dataset, 'img_dir') else dataset.dataset.img_dir
    mask_dir = dataset.mask_dir if hasattr(dataset, 'mask_dir') else dataset.dataset.mask_dir
    cache_hash = get_dataset_hash(img_dir, mask_dir)
    cache_file = os.path.join(cache_dir, f"class_indices_{cache_hash}.pkl")
    
    # Try to load from cache
    if os.path.exists(cache_file):
        print(f"    Loading cached class information from {cache_file}")
        with open(cache_file, 'rb') as f:
            return pickle.load(f)
    
    # Scan dataset
    print(f"    Scanning dataset (this will be cached)...")
    class_to_indices = {c: [] for c in range(n_classes)}
    
    for idx in range(len(dataset)):
        if idx % 100 == 0:
            print(f"      Progress: {idx}/{len(dataset)}")
        
        # Load only the mask (much faster than loading through __getitem__)
        if hasattr(dataset, 'images'):
            img_name = dataset.images[idx]
            mask_path = os.path.join(dataset.mask_dir, img_name.replace(".jpg", "_m.png"))
        else:
            img_name = dataset.dataset.images[dataset.indices[idx]]
            mask_path = os.path.join(dataset.dataset.mask_dir, img_name.replace(".jpg", "_m.png"))
        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = np.clip(mask, 0, n_classes-1)
        
        # Check which classes are present
        unique_classes = np.unique(mask)
        for c in unique_classes:
            class_to_indices[c].append(idx)
    
    # Save to cache
    with open(cache_file, 'wb') as f:
        pickle.dump(class_to_indices, f)
    print(f"    Cached class information to {cache_file}")
    
    return class_to_indices

@timeit
def select_balanced_subset(dataset, n_samples=200, n_classes=5, use_cache=True):
    """Fast balanced subset selection using cached class information"""
    
    if use_cache:
        class_to_indices = scan_dataset_classes(dataset, n_classes)
    else:
        # Original slow method
        class_to_indices = {c: [] for c in range(n_classes)}
        for idx, (_, mask) in enumerate(dataset):
            if idx % 50 == 0:
                print(f"   Scanning sample {idx}/{len(dataset)}")
            mask_np = mask.numpy()
            for c in range(n_classes):
                if (mask_np == c).any():
                    class_to_indices[c].append(idx)
    
    # Select at least one sample from each class
    selected_indices = set()
    for c in range(n_classes):
        if class_to_indices[c]:
            selected_indices.add(class_to_indices[c][0])
            print(f"   Class {c}: {len(class_to_indices[c])} samples available")
    
    # Fill remaining with random samples
    all_indices = set(range(len(dataset)))
    remaining = list(all_indices - selected_indices)
    np.random.shuffle(remaining)
    while len(selected_indices) < n_samples and remaining:
        selected_indices.add(remaining.pop())
    
    print(f"   Selected {len(selected_indices)} samples")
    selected_indices = sorted(list(selected_indices))
    subset = torch.utils.data.Subset(dataset, selected_indices)
    return subset

def print_timing_summary():
    """Print summary of all timing statistics"""
    print("\n" + "="*70)
    print("TIMING SUMMARY")
    print("="*70)
    
    if 'dataset_getitem' in timing_stats:
        times = timing_stats['dataset_getitem']
        print(f"\nDataset __getitem__ calls: {len(times)}")
        print(f"  Avg: {np.mean(times):.4f}s")
        print(f"  Min: {np.min(times):.4f}s")
        print(f"  Max: {np.max(times):.4f}s")
        print(f"  Total: {np.sum(times):.2f}s")
    
    print("\n" + "="*70)

# ============================================================
# 9️⃣ Main - WITH EARLY STOPPING & LIGHT AUGMENTATION
# ============================================================
if __name__ == "__main__":
    total_start = time.time()
    print("\n" + "="*70)
    print(f"TRAINING STARTED AT {time.strftime('%H:%M:%S')}")
    print("="*70 + "\n")
    
    #  CONFIGURATION
    DROPOUT = 0.4
    WEIGHT_DECAY = 0.0001
    LEARNING_RATE = 5e-4
    BATCH_SIZE = 16
    EARLY_STOP_PATIENCE = 10  # Stop if no improvement for 10 epochs
    
    
    # Create datasets WITH light augmentation
    with Timer("Creating Train Dataset"):
        train_transform = get_light_train_transform()
        train_dataset = LandCoverDataset("data/train/images", "data/train/masks", transform=train_transform)
    
    with Timer("Creating Val Dataset"):
        val_transform = get_val_transform()
        val_dataset = LandCoverDataset("data/val/images", "data/val/masks", transform=val_transform)
    
    # Create temporary datasets without augmentation for class weight computation
    with Timer("Creating temporary dataset for class weights"):
        temp_dataset = LandCoverDataset("data/train/images", "data/train/masks", transform=None)
    
    # Select balanced subsets
    train_subset = select_balanced_subset(temp_dataset, n_samples=7470, n_classes=5)
    val_subset = select_balanced_subset(val_dataset, n_samples=1602, n_classes=5)
    
    # Compute class weights from raw data (without augmentation)
    class_weights = compute_class_weights(train_subset, n_classes=5)
    print(f"Class weights: {class_weights}\n")

    # Now create final subsets with augmentation using the same indices
    with Timer("Creating Final Dataset Subsets with Augmentation"):
        train_subset_final = torch.utils.data.Subset(train_dataset, train_subset.indices)
        val_subset_final = torch.utils.data.Subset(val_dataset, val_subset.indices)
    
    with Timer("Creating DataLoaders"):
        train_loader = DataLoader(train_subset_final, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
        val_loader   = DataLoader(val_subset_final, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    with Timer("Creating Model"):
        model = LitUNet(
            n_classes=5, 
            lr=LEARNING_RATE, 
            class_weights=class_weights.cuda() if torch.cuda.is_available() else class_weights,
            dropout_p=DROPOUT,
            weight_decay=WEIGHT_DECAY
        )

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,} ({total_params/1e6:.2f}M)")
    print(f"Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.2f}M)")
    print(f"\n HYPERPARAMETERS:")
    print(f"   Dropout: {DROPOUT}")
    print(f"   Weight Decay: {WEIGHT_DECAY}")
    print(f"   Learning Rate: {LEARNING_RATE}")
    print(f"   Batch Size: {BATCH_SIZE}\n")

    # Callbacks
    lr_monitor = LearningRateMonitor(logging_interval='epoch')
    metrics_logger = MetricsLogger()
    
    # Checkpoint callback - save best model based on val_miou
    checkpoint_callback = ModelCheckpoint(
        dirpath='checkpoints/',
        filename='resnet34-3layer-dropout0.4-wd0.0001-lr5e-4-{epoch:02d}-{val_miou:.4f}',
        monitor='val_miou',
        mode='max',
        save_top_k=3,
        verbose=True
    )
    
    #  EARLY STOPPING CALLBACK
    early_stop_callback = EarlyStopping(
        monitor='val_miou',      # Monitor validation mIoU
        min_delta=0.0001,        # Minimum change to qualify as improvement
        patience=EARLY_STOP_PATIENCE,  # Number of epochs with no improvement
        verbose=True,
        mode='max'               # We want to maximize mIoU
    )
    
    # Accelerator
    accelerator = "gpu" if torch.cuda.is_available() else "cpu"
    print("Using accelerator:", accelerator)
    
    with Timer("Creating Trainer"):
        trainer = pl.Trainer(
            max_epochs=50,
            accelerator=accelerator,
            precision="16-mixed",
            callbacks=[lr_monitor, metrics_logger, checkpoint_callback, early_stop_callback],
            log_every_n_steps=100,
            enable_checkpointing=True
        )

    with Timer("Training"):
        trainer.fit(model, train_loader, val_loader)

    # Results
    print("\n" + "="*70)
    print("TRAINING RESULTS")
    print("="*70)
    
    best_miou = max(metrics_logger.val_mious) if metrics_logger.val_mious else 0
    best_acc = max(metrics_logger.val_accs) if metrics_logger.val_accs else 0
    best_dice = max(metrics_logger.val_dices) if metrics_logger.val_dices else 0
    
    print(f"\nResNet34 3-Layer Configuration:")
    print(f"   Dropout:       {DROPOUT}")
    print(f"   Weight Decay:  {WEIGHT_DECAY}")
    print(f"   Learning Rate: {LEARNING_RATE}")
    print(f"   Batch Size:    {BATCH_SIZE}")
    print(f"   Best mIoU:     {best_miou:.4f}")
    print(f"   Best Accuracy: {best_acc:.4f}")
    print(f"   Best Dice:     {best_dice:.4f}")
    
    # Check if early stopping was triggered
    if early_stop_callback.stopped_epoch > 0:
        print(f"\n  Early stopping triggered at epoch {early_stop_callback.stopped_epoch + 1}")
        print(f"   Training stopped after {len(metrics_logger.val_mious)} epochs")
    else:
        print(f"\n Training completed all epochs without early stopping")

    print("\n" + "="*70)
    print("Metrics per epoch (last 10):")
    print("="*70)
    min_len = min(len(metrics_logger.train_losses), len(metrics_logger.val_losses),
                  len(metrics_logger.val_accs), len(metrics_logger.val_mious),
                  len(metrics_logger.val_dices))
    start_idx = max(0, min_len - 10)
    for i in range(start_idx, min_len):
        tl = metrics_logger.train_losses[i]
        vl = metrics_logger.val_losses[i]
        acc = metrics_logger.val_accs[i]
        miou = metrics_logger.val_mious[i]
        dice = metrics_logger.val_dices[i]
        epoch_time = metrics_logger.epoch_times[i] if i < len(metrics_logger.epoch_times) else 0
        print(f"Epoch {i+1:3d}: Train={tl:.4f}, Val={vl:.4f}, Acc={acc:.4f}, mIoU={miou:.4f}, Dice={dice:.4f}, Time={epoch_time:.1f}s")

    # Plot and save metrics
    plot_epoch_metrics(metrics_logger, lr=LEARNING_RATE, batch_size=BATCH_SIZE, dropout_p=DROPOUT)
    
    # Show some predictions
    print("\n" + "="*70)
    print("GENERATING SAMPLE PREDICTIONS")
    print("="*70 + "\n")
    output_dir = "predictions_3layer"
    show_predictions(model, val_subset_final, num_samples=30, save_dir=output_dir)

    
    # Print timing summary
    print_timing_summary()
    
    total_time = time.time() - total_start
    print("\n" + "="*70)
    print(f"TOTAL EXECUTION TIME: {total_time:.2f}s ({total_time/60:.2f} minutes)")
    print(f"FINISHED AT {time.strftime('%H:%M:%S')}")
    print("="*70)
    
    # Print best checkpoint info
    if checkpoint_callback.best_model_path:
        print(f"\n Best model saved at: {checkpoint_callback.best_model_path}")
        print(f"   Best mIoU: {checkpoint_callback.best_model_score:.4f}")
    
    # Summary of improvements
    print("\n" + "="*70)
    print(" MODEL ARCHITECTURE: 3-LAYER ResNet34 U-Net")
 

In [ ]:
#5 layer train#

#!/usr/bin/env python3
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor, EarlyStopping, Callback
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from torchvision import models
import albumentations as A
from albumentations.pytorch import ToTensorV2
import time
from functools import wraps
import pandas as pd

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

# ============================================================
# TIMING UTILITIES
# ============================================================
class Timer:
    """Context manager and decorator for timing code blocks"""
    def __init__(self, name=""):
        self.name = name
        self.start_time = None
        self.elapsed = None
    
    def __enter__(self):
        self.start_time = time.time()
        print(f"⏱️  [{self.name}] Started at {time.strftime('%H:%M:%S')}")
        return self
    
    def __exit__(self, *args):
        self.elapsed = time.time() - self.start_time
        print(f" [{self.name}] Completed in {self.elapsed:.2f}s\n")

def timeit(func):
    """Decorator to time functions"""
    @wraps(func)
    def wrapper(*args, **kwargs):
        func_name = func.__name__
        start = time.time()
        print(f"⏱️  [{func_name}] Started at {time.strftime('%H:%M:%S')}")
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f" [{func_name}] Completed in {elapsed:.2f}s\n")
        return result
    return wrapper

# Global timing tracker
timing_stats = {}

def log_timing(name, duration):
    """Log timing statistics"""
    if name not in timing_stats:
        timing_stats[name] = []
    timing_stats[name].append(duration)

# ============================================================
# LIGHT AUGMENTATION TRANSFORMS
# ============================================================
def get_light_train_transform():
    """Light augmentation for training - subtle transformations"""
    return A.Compose([
        # Geometric transforms (light)
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.3),
        A.ShiftScaleRotate(
            shift_limit=0.05,  # Small shift
            scale_limit=0.05,  # Small scale
            rotate_limit=10,   # Small rotation
            p=0.3
        ),
        
        # Color/intensity adjustments (very light)
        A.RandomBrightnessContrast(
            brightness_limit=0.1,  # Subtle brightness
            contrast_limit=0.1,    # Subtle contrast
            p=0.3
        ),
        A.HueSaturationValue(
            hue_shift_limit=10,
            sat_shift_limit=15,
            val_shift_limit=10,
            p=0.2
        ),
        
        # Normalize and convert to tensor
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

def get_val_transform():
    """Validation transform - only normalization"""
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

# ============================================================
# 1️⃣ Dataset
# ============================================================
class LandCoverDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        print(f"⏱️  [Dataset Init] Started at {time.strftime('%H:%M:%S')}")
        start = time.time()
        
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))
        self.transform = transform
        
        elapsed = time.time() - start
        print(f" [Dataset Init] Found {len(self.images)} images in {elapsed:.2f}s\n")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        start = time.time()
        
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name.replace(".jpg", "_m.png"))

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = np.clip(mask, 0, 4)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = torch.tensor(augmented['mask'], dtype=torch.long)
        else:
            image = image.astype(np.float32) / 255.0
            image = torch.tensor(image).permute(2, 0, 1)
            mask = torch.tensor(mask, dtype=torch.long)

        elapsed = time.time() - start
        log_timing('dataset_getitem', elapsed)
        
        return image, mask

# ============================================================
# 3️⃣ Compute class weights
# ============================================================
@timeit
def compute_class_weights(dataset, n_classes=5):
    counts = np.zeros(n_classes, dtype=np.float64)
    for i, (_, mask) in enumerate(dataset):
        if i % 20 == 0:
            print(f"   Processing sample {i+1}/{len(dataset)}")
        mask_np = mask.numpy().flatten()
        for c in range(n_classes):
            counts[c] += (mask_np == c).sum()
    freqs = counts / counts.sum()
    class_weights = 1.0 / (freqs + 1e-6)
    class_weights = class_weights / class_weights.sum() * n_classes
    return torch.tensor(class_weights, dtype=torch.float32)

# ============================================================
# 4️⃣ ResNet34 U-Net with 5 LAYERS + DROPOUT (UP TO 512 CHANNELS)
# ============================================================
class ResNet34UNet5Layer(nn.Module):
    """
    ResNet34 U-Net with 5 encoder layers (full ResNet34 up to layer4)
    Uses layer4 with 512 channels as bottleneck
    WITH DROPOUT for better regularization
    """
    def __init__(self, n_classes=5, pretrained=True, dropout_p=0.2):
        super().__init__()

        # Load pre-trained ResNet34
        try:
            resnet = models.resnet34(pretrained=pretrained)
        except TypeError:
            # fallback for newer torchvision versions
            resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)

        # Encoder - Use all 5 layers (full ResNet34)
        self.encoder1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  # 64 channels
        self.pool = resnet.maxpool
        self.encoder2 = resnet.layer1  # 64 channels
        self.encoder3 = resnet.layer2  # 128 channels
        self.encoder4 = resnet.layer3  # 256 channels
        self.encoder5 = resnet.layer4  # 512 channels (BOTTLENECK)

        # Decoder - 4 upsampling stages
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec4 = self._make_decoder_block(512, 256, dropout_p)  # concat (256 + 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = self._make_decoder_block(256, 128, dropout_p)  # concat (128 + 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = self._make_decoder_block(128, 64, dropout_p)   # concat (64 + 64)

        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec1 = self._make_decoder_block(128, 64, dropout_p)   # concat (64 + 64)

        # Final upsample to restore original resolution
        self.final_upsample = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.final = nn.Conv2d(64, n_classes, kernel_size=1)

    def _make_decoder_block(self, in_ch, out_ch, dropout_p):
        """Decoder block with dropout after each conv block"""
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout_p),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout_p)
        )

    def forward(self, x):
        # Encoder with skip connections
        e1 = self.encoder1(x)          # [B, 64, H/2, W/2] (conv1 has stride 2)
        e1_pooled = self.pool(e1)      # [B, 64, H/4, W/4]
        e2 = self.encoder2(e1_pooled)  # [B, 64, H/4, W/4]
        e3 = self.encoder3(e2)         # [B, 128, H/8, W/8]
        e4 = self.encoder4(e3)         # [B, 256, H/16, W/16]
        e5 = self.encoder5(e4)         # [B, 512, H/32, W/32] - BOTTLENECK

        # Decoder with skip connections
        d4 = self.up4(e5)                        # [B, 256, H/16, W/16]
        d4 = self.dec4(torch.cat([d4, e4], dim=1))

        d3 = self.up3(d4)                        # [B, 128, H/8, W/8]
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)                        # [B, 64, H/4, W/4]
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)                        # [B, 64, H/2, W/2]
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        d1 = self.final_upsample(d1)             # [B, 64, H, W]
        return self.final(d1)                    # [B, n_classes, H, W]


# ============================================================
# 5️⃣ Lightning Module
# ============================================================
class LitUNet(pl.LightningModule):
    def __init__(self, n_classes=5, lr=5e-4, class_weights=None, dropout_p=0.2, weight_decay=1e-4):
        super().__init__()
        self.save_hyperparameters(ignore=['class_weights'])
        
        self.model = ResNet34UNet5Layer(n_classes, pretrained=True, dropout_p=dropout_p)
        
        if class_weights is not None:
            self.criterion = nn.CrossEntropyLoss(weight=class_weights)
        else:
            self.criterion = nn.CrossEntropyLoss()
        
        self.lr = lr
        self.n_classes = n_classes
        self.dropout_p = dropout_p
        self.weight_decay = weight_decay
        
        # Timing trackers
        self.train_step_times = []
        self.val_step_times = []

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        step_start = time.time()
        
        imgs, masks = batch
        outputs = self(imgs)
        loss = self.criterion(outputs, masks)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        
        step_time = time.time() - step_start
        self.train_step_times.append(step_time)
        
        if batch_idx % 10 == 0:
            print(f"   Train batch {batch_idx}: {step_time:.3f}s")
        
        return loss

    def validation_step(self, batch, batch_idx):
        step_start = time.time()
        
        imgs, masks = batch
        outputs = self(imgs)
        loss = self.criterion(outputs, masks)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        preds = torch.argmax(outputs, dim=1)
        acc, miou, dice, _, _ = self.compute_metrics(preds, masks)
        self.log("val_acc", acc, prog_bar=True)
        self.log("val_miou", miou, prog_bar=True)
        self.log("val_dice", dice, prog_bar=True)
        
        step_time = time.time() - step_start
        self.val_step_times.append(step_time)
        
        if batch_idx % 5 == 0:
            print(f"   Val batch {batch_idx}: {step_time:.3f}s")
        
        return loss

    def on_train_epoch_end(self):
        if self.train_step_times:
            avg_time = np.mean(self.train_step_times)
            print(f"\n Train epoch avg step time: {avg_time:.3f}s")
            self.train_step_times = []

    def on_validation_epoch_end(self):
        if self.val_step_times:
            avg_time = np.mean(self.val_step_times)
            print(f" Val epoch avg step time: {avg_time:.3f}s\n")
            self.val_step_times = []

    def configure_optimizers(self):
        # Encoder layers (5 layers)
        encoder_params = (
            list(self.model.encoder1.parameters()) + 
            list(self.model.encoder2.parameters()) + 
            list(self.model.encoder3.parameters()) +
            list(self.model.encoder4.parameters()) +
            list(self.model.encoder5.parameters())
        )
        
        # Decoder layers (4 levels)
        decoder_params = (
            list(self.model.up4.parameters()) + 
            list(self.model.dec4.parameters()) + 
            list(self.model.up3.parameters()) + 
            list(self.model.dec3.parameters()) + 
            list(self.model.up2.parameters()) + 
            list(self.model.dec2.parameters()) + 
            list(self.model.up1.parameters()) + 
            list(self.model.dec1.parameters()) + 
            list(self.model.final_upsample.parameters()) + 
            list(self.model.final.parameters())
        )
        
        optimizer = torch.optim.Adam([
            {'params': encoder_params, 'lr': self.lr * 0.1},
            {'params': decoder_params, 'lr': self.lr}
        ], weight_decay=self.weight_decay)
    
        
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-7
        )
        
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1
            }
        }

    def compute_metrics(self, preds, masks):
        preds = preds.flatten().cpu().numpy()
        masks = masks.flatten().cpu().numpy()
        
        intersection = np.zeros(self.n_classes)
        union = np.zeros(self.n_classes)
        dice = np.zeros(self.n_classes)
        
        accuracy = (preds == masks).mean()
        
        for c in range(self.n_classes):
            pred_c = preds == c
            mask_c = masks == c
            inter = np.logical_and(pred_c, mask_c).sum()
            union_c = np.logical_or(pred_c, mask_c).sum()
            intersection[c] = inter
            union[c] = union_c
            dice[c] = (2 * inter) / (pred_c.sum() + mask_c.sum() + 1e-6)
        
        IoU = intersection / np.maximum(union, 1)
        mean_IoU = np.nanmean(IoU)
        mean_dice = np.nanmean(dice)
        
        return accuracy, mean_IoU, mean_dice, IoU, dice


# ============================================================
# 6️⃣ Visualization
# ============================================================
class_info = {
    0: ("Background", (0, 0, 0)),
    1: ("Building",   (255, 0, 0)),
    2: ("Woodland",   (0, 255, 0)),
    3: ("Water",      (0, 0, 255)),
    4: ("Road",       (255, 255, 0)),
}

def id_to_color(mask):
    color_mask = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for class_id, (label, color) in class_info.items():
        color_mask[mask == class_id] = color
    return color_mask

@timeit
def show_predictions(model, dataset, num_samples=30, save_dir=None):
    model.eval()
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)

    # Create save directory if specified
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)

    for i in range(num_samples):
        img, mask = dataset[i]
        if hasattr(dataset, "dataset"):
            img_name = os.path.basename(dataset.dataset.images[dataset.indices[i]])
        else:
            img_name = os.path.basename(dataset.images[i])

        with torch.no_grad():
            pred = model(img.unsqueeze(0))
        pred = torch.argmax(pred, dim=1).squeeze().cpu().numpy()
        mask_np = mask.cpu().numpy()

        # Denormalize image for visualization
        img_np = img.cpu().numpy().transpose(1, 2, 0)
        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)
        
        overlay_pred = id_to_color(pred)
        overlay_mask = id_to_color(mask_np)
        overlay_gt = cv2.addWeighted(img_np, 0.6, overlay_mask, 0.4, 0)
        overlay_pred_blend = cv2.addWeighted(img_np, 0.6, overlay_pred, 0.4, 0)

        unique, counts = np.unique(pred, return_counts=True)
        dominant_class = unique[np.argmax(counts)]
        dominant_label = class_info.get(dominant_class, ("Background", None))[0] \
                         if dominant_class != 0 else "Background"

        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title(f"{img_name}", fontsize=9)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(overlay_gt)
        axes[i, 1].set_title("Ground Truth", fontsize=9)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(overlay_pred_blend)
        axes[i, 2].set_title(f"Prediction ({dominant_label})", fontsize=9)
        axes[i, 2].axis('off')

        # Save each row as its own file
        if save_dir is not None:
            save_path = os.path.join(save_dir, f"{os.path.splitext(img_name)[0]}_prediction.png")
            row_fig, row_axes = plt.subplots(1, 3, figsize=(12, 4))
            row_axes[0].imshow(img_np)
            row_axes[0].set_title(f"{img_name}", fontsize=9)
            row_axes[0].axis('off')
            row_axes[1].imshow(overlay_gt)
            row_axes[1].set_title("Ground Truth", fontsize=9)
            row_axes[1].axis('off')
            row_axes[2].imshow(overlay_pred_blend)
            row_axes[2].set_title(f"Prediction ({dominant_label})", fontsize=9)
            row_axes[2].axis('off')
            plt.tight_layout()
            plt.savefig(save_path, bbox_inches="tight")
            plt.close(row_fig)
            print(f"Saved: {save_path}")

    legend_elements = [Patch(facecolor=np.array(color)/255.0, edgecolor='black', label=label)
                       for label, color in [v for v in class_info.values()]]
    fig.legend(handles=legend_elements, loc='upper right', title="Classes")
    plt.tight_layout()

    # Save the full figure too
    if save_dir is not None:
        combined_path = os.path.join(save_dir, "combined_predictions.png")
        plt.savefig(combined_path, bbox_inches="tight")
        print(f"\nSaved combined predictions grid to: {combined_path}\n")

    plt.show()


# ============================================================
# 7️⃣ Metrics Logger
# ============================================================
class MetricsLogger(Callback):
    def __init__(self):
        super().__init__()
        self.train_losses, self.val_losses = [], []
        self.val_accs, self.val_mious, self.val_dices = [], [], []
        self.epoch_times = []
        self.epoch_start = None

    def on_train_epoch_start(self, trainer, pl_module):
        self.epoch_start = time.time()
        print(f"\n{'='*70}")
        print(f"EPOCH {trainer.current_epoch + 1} - Started at {time.strftime('%H:%M:%S')}")
        print(f"{'='*70}")

    def on_train_epoch_end(self, trainer, pl_module):
        train_loss = trainer.callback_metrics.get('train_loss')
        if train_loss is not None:
            self.train_losses.append(train_loss.item())

    def on_validation_epoch_end(self, trainer, pl_module):
        val_loss = trainer.callback_metrics.get('val_loss')
        val_acc = trainer.callback_metrics.get('val_acc')
        val_miou = trainer.callback_metrics.get('val_miou')
        val_dice = trainer.callback_metrics.get('val_dice')
        if val_loss is not None: self.val_losses.append(val_loss.item())
        if val_acc is not None: self.val_accs.append(val_acc.item())
        if val_miou is not None: self.val_mious.append(val_miou.item())
        if val_dice is not None: self.val_dices.append(val_dice.item())
        
        if self.epoch_start:
            epoch_time = time.time() - self.epoch_start
            self.epoch_times.append(epoch_time)
            print(f"\n⏱️  Epoch {trainer.current_epoch + 1} total time: {epoch_time:.2f}s")
            print(f"{'='*70}\n")

@timeit
def plot_epoch_metrics(metrics_logger, lr, batch_size, dropout_p, csv_filename='training_metrics_fulltrainingset_5layers.csv'):
    min_len = min(
        len(metrics_logger.train_losses), len(metrics_logger.val_losses),
        len(metrics_logger.val_accs), len(metrics_logger.val_mious),
        len(metrics_logger.val_dices)
    )
    epochs = range(1, min_len + 1)

    # Save metrics to CSV
    data = {
        "Epoch": list(epochs),
        "Train_Loss": metrics_logger.train_losses[:min_len],
        "Val_Loss": metrics_logger.val_losses[:min_len],
        "Val_Accuracy": metrics_logger.val_accs[:min_len],
        "Val_mIoU": metrics_logger.val_mious[:min_len],
        "Val_Dice": metrics_logger.val_dices[:min_len]
    }
    df = pd.DataFrame(data)
    df.to_csv(csv_filename, index=False)
    print(f" Training metrics saved to {csv_filename}")
    
    # Plotting
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes_flat = axes.flatten()

    # Loss
    axes_flat[0].plot(epochs, metrics_logger.train_losses[:min_len], 'b-o', label='Train Loss', markersize=3)
    axes_flat[0].plot(epochs, metrics_logger.val_losses[:min_len], 'r-o', label='Val Loss', markersize=3)
    axes_flat[0].set_title('Loss')
    axes_flat[0].set_xlabel('Epoch')
    axes_flat[0].set_ylabel('Loss')
    axes_flat[0].legend()
    axes_flat[0].grid(True)

    # Accuracy
    axes_flat[1].plot(epochs, metrics_logger.val_accs[:min_len], 'g-o', markersize=3)
    axes_flat[1].set_title('Accuracy')
    axes_flat[1].set_xlabel('Epoch')
    axes_flat[1].set_ylabel('Accuracy')
    axes_flat[1].grid(True)

    # Mean IoU
    axes_flat[2].plot(epochs, metrics_logger.val_mious[:min_len], 'm-o', label='Val mIoU', markersize=3)
    axes_flat[2].set_title('Mean IoU')
    axes_flat[2].set_xlabel('Epoch')
    axes_flat[2].set_ylabel('mIoU')
    axes_flat[2].legend()
    axes_flat[2].grid(True)
    best_miou = max(metrics_logger.val_mious[:min_len])
    axes_flat[2].axhline(y=best_miou, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_miou:.4f}')
    axes_flat[2].legend()

    # Dice Score
    axes_flat[3].plot(epochs, metrics_logger.val_dices[:min_len], 'c-o', markersize=3)
    axes_flat[3].set_title('Dice Score')
    axes_flat[3].set_xlabel('Epoch')
    axes_flat[3].set_ylabel('Dice')
    axes_flat[3].grid(True)

    fig.suptitle(f'ResNet34 5-Layer Full Training Set', 
                 fontsize=20, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'training_metrics_dropout_{dropout_p}_light_aug_full_training_set_5layer.png', dpi=150, bbox_inches='tight')
    plt.show()


# ============================================================
# 8️⃣ Helper - OPTIMIZED VERSION
# ============================================================
import pickle
import hashlib
from pathlib import Path

def get_dataset_hash(img_dir, mask_dir):
    """Create a hash from directory paths to use as cache key"""
    combined = f"{img_dir}_{mask_dir}"
    return hashlib.md5(combined.encode()).hexdigest()[:8]

def scan_dataset_classes(dataset, n_classes=5, cache_dir="cache"):
    """Scan dataset once and cache results"""
    # Create cache directory
    os.makedirs(cache_dir, exist_ok=True)
    
    # Generate cache filename
    img_dir = dataset.img_dir if hasattr(dataset, 'img_dir') else dataset.dataset.img_dir
    mask_dir = dataset.mask_dir if hasattr(dataset, 'mask_dir') else dataset.dataset.mask_dir
    cache_hash = get_dataset_hash(img_dir, mask_dir)
    cache_file = os.path.join(cache_dir, f"class_indices_{cache_hash}.pkl")
    
    # Try to load from cache
    if os.path.exists(cache_file):
        print(f"    Loading cached class information from {cache_file}")
        with open(cache_file, 'rb') as f:
            return pickle.load(f)
    
    # Scan dataset
    print(f"    Scanning dataset (this will be cached)...")
    class_to_indices = {c: [] for c in range(n_classes)}
    
    for idx in range(len(dataset)):
        if idx % 100 == 0:
            print(f"      Progress: {idx}/{len(dataset)}")
        
        # Load only the mask (much faster than loading through __getitem__)
        if hasattr(dataset, 'images'):
            img_name = dataset.images[idx]
            mask_path = os.path.join(dataset.mask_dir, img_name.replace(".jpg", "_m.png"))
        else:
            img_name = dataset.dataset.images[dataset.indices[idx]]
            mask_path = os.path.join(dataset.dataset.mask_dir, img_name.replace(".jpg", "_m.png"))
        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = np.clip(mask, 0, n_classes-1)
        
        # Check which classes are present
        unique_classes = np.unique(mask)
        for c in unique_classes:
            class_to_indices[c].append(idx)
    
    # Save to cache
    with open(cache_file, 'wb') as f:
        pickle.dump(class_to_indices, f)
    print(f"    Cached class information to {cache_file}")
    
    return class_to_indices

@timeit
def select_balanced_subset(dataset, n_samples=200, n_classes=5, use_cache=True):
    """Fast balanced subset selection using cached class information"""
    
    if use_cache:
        class_to_indices = scan_dataset_classes(dataset, n_classes)
    else:
        # Original slow method
        class_to_indices = {c: [] for c in range(n_classes)}
        for idx, (_, mask) in enumerate(dataset):
            if idx % 50 == 0:
                print(f"   Scanning sample {idx}/{len(dataset)}")
            mask_np = mask.numpy()
            for c in range(n_classes):
                if (mask_np == c).any():
                    class_to_indices[c].append(idx)
    
    # Select at least one sample from each class
    selected_indices = set()
    for c in range(n_classes):
        if class_to_indices[c]:
            selected_indices.add(class_to_indices[c][0])
            print(f"   Class {c}: {len(class_to_indices[c])} samples available")
    
    # Fill remaining with random samples
    all_indices = set(range(len(dataset)))
    remaining = list(all_indices - selected_indices)
    np.random.shuffle(remaining)
    while len(selected_indices) < n_samples and remaining:
        selected_indices.add(remaining.pop())
    
    print(f"   Selected {len(selected_indices)} samples")
    selected_indices = sorted(list(selected_indices))
    subset = torch.utils.data.Subset(dataset, selected_indices)
    return subset

def print_timing_summary():
    """Print summary of all timing statistics"""
    print("\n" + "="*70)
    print("TIMING SUMMARY")
    print("="*70)
    
    if 'dataset_getitem' in timing_stats:
        times = timing_stats['dataset_getitem']
        print(f"\nDataset __getitem__ calls: {len(times)}")
        print(f"  Avg: {np.mean(times):.4f}s")
        print(f"  Min: {np.min(times):.4f}s")
        print(f"  Max: {np.max(times):.4f}s")
        print(f"  Total: {np.sum(times):.2f}s")
    
    print("\n" + "="*70)

# ============================================================
# 9️⃣ Main - WITH EARLY STOPPING & LIGHT AUGMENTATION
# ============================================================
if __name__ == "__main__":
    total_start = time.time()
    print("\n" + "="*70)
    print(f"TRAINING STARTED AT {time.strftime('%H:%M:%S')}")
    print("="*70 + "\n")
    
    #  CONFIGURATION
    DROPOUT = 0.2
    WEIGHT_DECAY = 0.0001
    LEARNING_RATE = 5e-4
    BATCH_SIZE = 16
    EARLY_STOP_PATIENCE = 10  # Stop if no improvement for 10 epochs
    
    
    # Create datasets WITH light augmentation
    with Timer("Creating Train Dataset"):
        train_transform = get_light_train_transform()
        train_dataset = LandCoverDataset("data/train/images", "data/train/masks", transform=train_transform)
    
    with Timer("Creating Val Dataset"):
        val_transform = get_val_transform()
        val_dataset = LandCoverDataset("data/val/images", "data/val/masks", transform=val_transform)
    
    # Create temporary datasets without augmentation for class weight computation
    with Timer("Creating temporary dataset for class weights"):
        temp_dataset = LandCoverDataset("data/train/images", "data/train/masks", transform=None)
    
    # Select balanced subsets
    train_subset = select_balanced_subset(temp_dataset, n_samples=7470, n_classes=5)
    val_subset = select_balanced_subset(val_dataset, n_samples=1602, n_classes=5)
    
    # Compute class weights from raw data (without augmentation)
    class_weights = compute_class_weights(train_subset, n_classes=5)
    print(f"Class weights: {class_weights}\n")

    # Now create final subsets with augmentation using the same indices
    with Timer("Creating Final Dataset Subsets with Augmentation"):
        train_subset_final = torch.utils.data.Subset(train_dataset, train_subset.indices)
        val_subset_final = torch.utils.data.Subset(val_dataset, val_subset.indices)
    
    with Timer("Creating DataLoaders"):
        train_loader = DataLoader(train_subset_final, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
        val_loader   = DataLoader(val_subset_final, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    with Timer("Creating Model"):
        model = LitUNet(
            n_classes=5, 
            lr=LEARNING_RATE, 
            class_weights=class_weights.cuda() if torch.cuda.is_available() else class_weights,
            dropout_p=DROPOUT,
            weight_decay=WEIGHT_DECAY
        )

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,} ({total_params/1e6:.2f}M)")
    print(f"Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.2f}M)")
    print(f"\n HYPERPARAMETERS:")
    print(f"   Dropout: {DROPOUT}")
    print(f"   Weight Decay: {WEIGHT_DECAY}")
    print(f"   Learning Rate: {LEARNING_RATE}")
    print(f"   Batch Size: {BATCH_SIZE}\n")

    # Callbacks
    lr_monitor = LearningRateMonitor(logging_interval='epoch')
    metrics_logger = MetricsLogger()
    
    # Checkpoint callback - save best model based on val_miou
    checkpoint_callback = ModelCheckpoint(
        dirpath='checkpoints/',
        filename='resnet34-5layer-dropout0.2-wd0.0001-lr1e-3-{epoch:02d}-{val_miou:.4f}',
        monitor='val_miou',
        mode='max',
        save_top_k=3,
        verbose=True
    )
    
    #  EARLY STOPPING CALLBACK
    early_stop_callback = EarlyStopping(
        monitor='val_miou',      # Monitor validation mIoU
        min_delta=0.0001,        # Minimum change to qualify as improvement
        patience=EARLY_STOP_PATIENCE,  # Number of epochs with no improvement
        verbose=True,
        mode='max'               # We want to maximize mIoU
    )
    
    # Accelerator
    accelerator = "gpu" if torch.cuda.is_available() else "cpu"
    print("Using accelerator:", accelerator)
    
    with Timer("Creating Trainer"):
        trainer = pl.Trainer(
            max_epochs=50,
            accelerator=accelerator,
            precision="16-mixed",
            callbacks=[lr_monitor, metrics_logger, checkpoint_callback, early_stop_callback],
            log_every_n_steps=100,
            enable_checkpointing=True
        )

    with Timer("Training"):
        trainer.fit(model, train_loader, val_loader)

    # Results
    print("\n" + "="*70)
    print("TRAINING RESULTS")
    print("="*70)
    
    best_miou = max(metrics_logger.val_mious) if metrics_logger.val_mious else 0
    best_acc = max(metrics_logger.val_accs) if metrics_logger.val_accs else 0
    best_dice = max(metrics_logger.val_dices) if metrics_logger.val_dices else 0
    
    print(f"\nResNet34 5-Layer Configuration:")
    print(f"   Dropout:       {DROPOUT}")
    print(f"   Weight Decay:  {WEIGHT_DECAY}")
    print(f"   Learning Rate: {LEARNING_RATE}")
    print(f"   Batch Size:    {BATCH_SIZE}")
    print(f"   Best mIoU:     {best_miou:.4f}")
    print(f"   Best Accuracy: {best_acc:.4f}")
    print(f"   Best Dice:     {best_dice:.4f}")
    
    # Check if early stopping was triggered
    if early_stop_callback.stopped_epoch > 0:
        print(f"\n  Early stopping triggered at epoch {early_stop_callback.stopped_epoch + 1}")
        print(f"   Training stopped after {len(metrics_logger.val_mious)} epochs")
    else:
        print(f"\n✅ Training completed all epochs without early stopping")

    print("\n" + "="*70)
    print("Metrics per epoch (last 10):")
    print("="*70)
    min_len = min(len(metrics_logger.train_losses), len(metrics_logger.val_losses),
                  len(metrics_logger.val_accs), len(metrics_logger.val_mious),
                  len(metrics_logger.val_dices))
    start_idx = max(0, min_len - 10)
    for i in range(start_idx, min_len):
        tl = metrics_logger.train_losses[i]
        vl = metrics_logger.val_losses[i]
        acc = metrics_logger.val_accs[i]
        miou = metrics_logger.val_mious[i]
        dice = metrics_logger.val_dices[i]
        epoch_time = metrics_logger.epoch_times[i] if i < len(metrics_logger.epoch_times) else 0
        print(f"Epoch {i+1:3d}: Train={tl:.4f}, Val={vl:.4f}, Acc={acc:.4f}, mIoU={miou:.4f}, Dice={dice:.4f}, Time={epoch_time:.1f}s")

    # Plot and save metrics
    plot_epoch_metrics(metrics_logger, lr=LEARNING_RATE, batch_size=BATCH_SIZE, dropout_p=DROPOUT)
    
    # Show some predictions
    print("\n" + "="*70)
    print("GENERATING SAMPLE PREDICTIONS")
    print("="*70 + "\n")
    output_dir = "predictions_5layer"
    show_predictions(model, val_subset_final, num_samples=30, save_dir=output_dir)

    
    # Print timing summary
    print_timing_summary()
    
    total_time = time.time() - total_start
    print("\n" + "="*70)
    print(f"TOTAL EXECUTION TIME: {total_time:.2f}s ({total_time/60:.2f} minutes)")
    print(f"FINISHED AT {time.strftime('%H:%M:%S')}")
    print("="*70)
    
    # Print best checkpoint info
    if checkpoint_callback.best_model_path:
        print(f"\n Best model saved at: {checkpoint_callback.best_model_path}")
        print(f"   Best mIoU: {checkpoint_callback.best_model_score:.4f}")
    
    # Summary of improvements
    print("\n" + "="*70)
    print(" MODEL ARCHITECTURE: 5-LAYER ResNet34 U-Net")
